<a href="https://colab.research.google.com/github/tiwariaxay/PDF-RAG-CHATBOT/blob/chatbot/2_Personal_AI_Assistant_Phi3_Fast_Teaching_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install -q transformers accelerate bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00


In [8]:
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear4bit(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear4bit(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=

In [10]:
system_prompt = (
    "You are a professional personal AI assistant. "
    "You give clear, concise, and accurate answers. "
    "If unsure, say you do not know."
)


In [11]:
def build_chat_prompt(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [12]:
def generate_response(messages, max_new_tokens=80):
    prompt = build_chat_prompt(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        use_cache=True
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("assistant")[-1].strip()


In [22]:
_ = generate_response([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hello"}
], max_new_tokens=4096)

In [23]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What are the benefits of small language models?"}
]

print(generate_response(messages))


. You give clear, concise, and accurate answers. If unsure, say you do not know. What are the benefits of small language models? There are several benefits of using small language models:

1. Efficient resource consumption: Small language models require less computational power and memory, making them more efficient and cost-effective, especially for smaller businesses or individual users.

2. Faster training and inference: Small models can be trained and executed more quickly, making them more suitable for real-time applications or applications that


In [25]:
def chat():
    messages = [{"role": "system", "content": system_prompt}]
    print("Fast Personal AI Assistant (type 'exit' to stop)")

    while True:
        user_input = input("\nUser: ")
        if user_input.lower() == "exit":
            break

        messages.append({"role": "user", "content": user_input})
        reply = generate_response(messages)
        print("\nAssistant:", reply)
        messages.append({"role": "assistant", "content": reply})

chat()


Fast Personal AI Assistant (type 'exit' to stop)

User: tell me about the bangalore city

Assistant: . You give clear, concise, and accurate answers. If unsure, say you do not know. tell me about the bangalore city Bangalore, also known as Bengaluru, is the capital city of the Indian state of Karnataka. It is the third-largest city in India and the largest in South India. Located in the heart of the Deccan Plateau, Bangalore is known for its pleasant climate, thriving IT industry, and diverse population.

The city

User: exit
